ORIGIN


In [ ]:
# """
# PIPELINE DỰ ĐOÁN RETURN CỔ PHIẾU VN – DỰ ÁN A (PRE-REGISTERED)
# =============================================================
# Đã sửa lỗi KeyError khi truy cập sim_days không liên tục.
# - Tự động map cột time->date, symbol->ticker.
# - Loại bỏ VNINDEX khỏi universe.
# - Cảnh báo nếu crisis fold 2026 chạy sau khi dữ liệu đã có.
# - PSR tính bằng công thức gốc (mở).
# """

# import numpy as np
# import pandas as pd
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dropout, Dense
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping
# from sklearn.preprocessing import StandardScaler
# from scipy.stats import norm
# import random
# import os
# import warnings
# from datetime import datetime
# warnings.filterwarnings('ignore')

# # ====================== CÀI ĐẶT TÁI LẬP ======================
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# tf.random.set_seed(SEED)
# os.environ['TF_DETERMINISTIC_OPS'] = '1'
# os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# # ====================== THAM SỐ PRE-REGISTERED ======================
# DATA_START = '2016-07-05'
# TARGET_HORIZON = 5
# SEQUENCE_LENGTH = 20
# TOP_UNIVERSE_FRAC = 0.5
# TOP_K_HOLD = 3
# REBALANCE_FREQ = 5
# TRADING_FEE = 0.004
# WINSORIZE_QUANTILES = (0.01, 0.99)
# MDD_STOP = -0.20
# PSR_THRESHOLD = 0.95
# EXCLUDED_TICKERS = ['VNINDEX']

# LSTM_UNITS = 64
# LSTM_LAYERS = 3
# DROPOUT = 0.2
# LR = 0.001
# BATCH_SIZE = 32
# MAX_EPOCHS = 200
# EARLY_STOP_PATIENCE = 10
# VAL_SPLIT = 0.2

# FOLDS_5 = [
#     ('2018-12-31', '2019-01-02', '2020-05-29'),
#     ('2020-05-29', '2020-06-01', '2021-10-29'),
#     ('2021-10-29', '2021-11-01', '2023-03-31'),
#     ('2023-03-31', '2023-04-03', '2024-08-30'),
#     ('2024-08-30', '2024-09-02', '2025-12-31')
# ]
# CRISIS_FOLDS = [
#     ('2019-12-31', '2020-03-01', '2020-06-30', 'covid_2020'),
#     ('2025-12-31', '2026-01-01', '2026-03-31', 'crash_2026')
# ]

# # ====================== TIỆN ÍCH ======================
# def load_and_prepare_data(filepath='merged_long_format.csv'):
#     first_row = pd.read_csv(filepath, nrows=0)
#     date_col = 'time' if 'time' in first_row.columns else 'date'
#     df = pd.read_csv(filepath, parse_dates=[date_col])
#     rename_map = {}
#     if 'time' in df.columns:
#         rename_map['time'] = 'date'
#     if 'symbol' in df.columns:
#         rename_map['symbol'] = 'ticker'
#     if rename_map:
#         df.rename(columns=rename_map, inplace=True)
#     df = df[(df['is_trading'] == 1) & (df['volume'] > 0)]
#     df['date'] = pd.to_datetime(df['date'])
#     df = df[df['date'] >= DATA_START]
#     df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
#     return df

# def get_trading_days(df):
#     days = df['date'].drop_duplicates().sort_values().reset_index(drop=True)
#     return days

# def nearest_trading_day(target, days, how='before'):
#     target = pd.Timestamp(target)
#     if how == 'before':
#         return days[days <= target].iloc[-1]
#     else:
#         return days[days >= target].iloc[0]

# def add_features(df):
#     df = df.copy()
#     df['return_1d'] = df.groupby('ticker')['close'].pct_change()
#     df['return_5d_lag'] = df.groupby('ticker')['close'].pct_change(5)
#     df['ma5'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(5).mean())
#     df['ma10'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(10).mean())
#     df['ma20'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
#     df['volatility_5d'] = df.groupby('ticker')['return_1d'].transform(lambda x: x.rolling(5).std())
#     df['volume_ma5'] = df.groupby('ticker')['volume'].transform(lambda x: x.rolling(5).mean())
#     df['high_low_ratio'] = df['high'] / df['low']
#     df['close_open_ratio'] = df['close'] / df['open']
#     df['dollar_volume'] = df['close'] * df['volume']
#     df['target_raw'] = df.groupby('ticker')['close'].shift(-TARGET_HORIZON) / df['close'] - 1
#     return df

# def prepare_training_data(df_features, tickers, train_dates, sequence_len=SEQUENCE_LENGTH):
#     feature_cols = [c for c in df_features.columns if c not in
#                     ['target_raw', 'target', 'dollar_volume', 'ticker', 'date']]
#     X_list, y_list = [], []
#     for tic in tickers:
#         sub = df_features[df_features['ticker'] == tic].sort_values('date')
#         sub = sub[sub['date'].isin(train_dates)]
#         sub = sub.dropna(subset=feature_cols + ['target'])
#         arr_f = sub[feature_cols].values
#         arr_t = sub['target'].values
#         for i in range(sequence_len, len(sub)):
#             X_list.append(arr_f[i-sequence_len:i])
#             y_list.append(arr_t[i])
#     return np.array(X_list), np.array(y_list).reshape(-1, 1)

# def build_lstm(input_shape):
#     model = Sequential()
#     model.add(LSTM(LSTM_UNITS, return_sequences=True, input_shape=input_shape))
#     model.add(Dropout(DROPOUT))
#     for _ in range(LSTM_LAYERS - 2):
#         model.add(LSTM(LSTM_UNITS, return_sequences=True))
#         model.add(Dropout(DROPOUT))
#     model.add(LSTM(LSTM_UNITS, return_sequences=False))
#     model.add(Dropout(DROPOUT))
#     model.add(Dense(1))
#     model.compile(optimizer=Adam(learning_rate=LR), loss='mse')
#     return model

# def filter_universe(df_features, date, trading_days, top_frac=TOP_UNIVERSE_FRAC, lookback=20):
#     idx = trading_days[trading_days <= date].index[-lookback:]
#     recent_dates = trading_days.iloc[idx]
#     recent = df_features[df_features['date'].isin(recent_dates)]
#     recent = recent[~recent['ticker'].isin(EXCLUDED_TICKERS)]
#     avg_dvol = recent.groupby('ticker')['dollar_volume'].mean().sort_values(ascending=False)
#     n_select = int(np.ceil(len(avg_dvol) * top_frac))
#     return list(avg_dvol.head(n_select).index)

# def predict_returns_for_date(model, scaler, df_features, tickers, date, sequence_len=SEQUENCE_LENGTH):
#     feature_cols = [c for c in df_features.columns if c not in
#                     ['target_raw', 'target', 'dollar_volume', 'ticker', 'date']]
#     preds = {}
#     for tic in tickers:
#         sub = df_features[(df_features['ticker'] == tic) & (df_features['date'] <= date)].tail(sequence_len)
#         if len(sub) < sequence_len:
#             preds[tic] = -np.inf
#             continue
#         seq = sub[feature_cols].values
#         seq_scaled = scaler.transform(seq)
#         pred = model.predict(seq_scaled[np.newaxis, ...], verbose=0)[0, 0]
#         preds[tic] = pred
#     return preds

# def run_backtest(df_features, model, scaler, tickers_all, start_date, end_date, trading_days,
#                  top_frac=TOP_UNIVERSE_FRAC, top_k=TOP_K_HOLD, fee=TRADING_FEE,
#                  rebalance_freq=REBALANCE_FREQ, initial_cap=1.0):
#     # SỬA LỖI KEYERROR: reset index để đảm bảo liên tục 0..len-1
#     sim_days = trading_days[(trading_days >= start_date) & (trading_days <= end_date)].reset_index(drop=True)
#     rebalance_dates = set(sim_days[::rebalance_freq])
#     portfolio_returns = []
#     turnover_log = []
#     corr_log = []
#     holdings = {}
#     prev_value = initial_cap

#     for i, date in enumerate(sim_days):
#         if i == 0:
#             daily_ret = 0.0
#             do_rebalance = True
#         else:
#             current_value = 0.0
#             for tic, shares in holdings.items():
#                 # Lấy giá hôm nay
#                 price_series = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
#                 if not price_series.empty:
#                     current_value += shares * price_series.iloc[0]
#                 else:
#                     # Không có giá, dùng giá hôm trước (sim_days đã reset index nên dùng được i-1)
#                     prev_price_series = df_features[(df_features['ticker']==tic) & (df_features['date']==sim_days.iloc[i-1])]['close']
#                     if not prev_price_series.empty:
#                         current_value += shares * prev_price_series.iloc[0]
#             daily_ret = (current_value / prev_value) - 1
#             prev_value = current_value
#             do_rebalance = (date in rebalance_dates)

#         if do_rebalance:
#             universe = filter_universe(df_features, date, trading_days, top_frac)
#             if len(universe) < top_k:
#                 selected = universe
#             else:
#                 preds = predict_returns_for_date(model, scaler, df_features, universe, date)
#                 sorted_preds = sorted(preds.items(), key=lambda x: x[1], reverse=True)
#                 selected = [t[0] for t in sorted_preds[:top_k]]

#             old_weights = {}
#             if holdings:
#                 total_val = prev_value
#                 for tic, shares in holdings.items():
#                     price_row = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
#                     if not price_row.empty:
#                         old_weights[tic] = shares * price_row.iloc[0] / total_val
#             new_weights = {t: 1.0/top_k for t in selected}
#             all_tickers = set(list(old_weights.keys()) + list(new_weights.keys()))
#             turnover = 0.5 * sum(abs(new_weights.get(t,0) - old_weights.get(t,0)) for t in all_tickers)
#             fee_cost = turnover * prev_value * fee
#             prev_value -= fee_cost
#             turnover_log.append((date, turnover))

#             holdings = {}
#             for tic in selected:
#                 price_row = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
#                 if not price_row.empty:
#                     price = price_row.iloc[0]
#                     holdings[tic] = (prev_value * new_weights[tic]) / price

#             if len(selected) >= 2:
#                 rets = []
#                 for tic in selected:
#                     sub = df_features[(df_features['ticker']==tic) & (df_features['date']<=date)].tail(20)
#                     if len(sub) >= 5:
#                         rets.append(sub['return_1d'].dropna().values[-20:])
#                     else:
#                         rets.append(np.zeros(20))
#                 if rets:
#                     corr_mat = np.corrcoef(rets)
#                     avg_corr = (corr_mat.sum() - len(selected)) / (len(selected)*(len(selected)-1))
#                     corr_log.append((date, avg_corr))

#         portfolio_returns.append(daily_ret)

#     res = pd.DataFrame({'date': sim_days, 'daily_return': portfolio_returns})
#     return res, turnover_log, corr_log

# def run_baseline_backtest(df_features, start_date, end_date, trading_days, top_frac=TOP_UNIVERSE_FRAC,
#                           fee=TRADING_FEE, rebalance_freq=REBALANCE_FREQ, initial_cap=1.0):
#     sim_days = trading_days[(trading_days >= start_date) & (trading_days <= end_date)].reset_index(drop=True)
#     rebalance_dates = set(sim_days[::rebalance_freq])
#     portfolio_returns = []
#     prev_value = initial_cap
#     holdings = {}
#     for i, date in enumerate(sim_days):
#         if i == 0:
#             do_rebalance = True
#             daily_ret = 0.0
#             portfolio_returns.append(daily_ret)
#         else:
#             current_value = 0.0
#             for tic, shares in holdings.items():
#                 price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
#                 if not price.empty:
#                     current_value += shares * price.iloc[0]
#             daily_ret = (current_value / prev_value) - 1
#             portfolio_returns.append(daily_ret)
#             prev_value = current_value
#             do_rebalance = (date in rebalance_dates)
#         if do_rebalance:
#             universe = filter_universe(df_features, date, trading_days, top_frac)
#             k = len(universe)
#             if holdings:
#                 old_weights = {}
#                 total_val = prev_value
#                 for tic, shares in holdings.items():
#                     price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
#                     if not price.empty:
#                         old_weights[tic] = shares * price.iloc[0] / total_val
#                 new_weights = {t: 1.0/k for t in universe}
#                 all_tickers = set(list(old_weights.keys()) + list(new_weights.keys()))
#                 turnover = 0.5 * sum(abs(new_weights.get(t,0) - old_weights.get(t,0)) for t in all_tickers)
#                 fee_cost = turnover * prev_value * fee
#                 prev_value -= fee_cost
#             holdings = {}
#             for tic in universe:
#                 price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
#                 if not price.empty:
#                     holdings[tic] = (prev_value / k) / price.iloc[0]
#     return pd.DataFrame({'date': sim_days.iloc[1:], 'daily_return': portfolio_returns[1:]})

# def concatenate_fold_returns(fold_results):
#     all_rets = []
#     for ret_df, _, _ in fold_results:
#         all_rets.append(ret_df['daily_return'])
#     return pd.concat(all_rets, ignore_index=True)

# def compute_psr(returns_series, benchmark_return=0.0):
#     rets = returns_series.dropna().values
#     n = len(rets)
#     if n < 2:
#         return np.nan
#     sr = np.mean(rets) / np.std(rets, ddof=1) if np.std(rets, ddof=1) != 0 else 0.0
#     skew = pd.Series(rets).skew()
#     kurt = pd.Series(rets).kurtosis() + 3
#     numerator = (sr - benchmark_return) * np.sqrt(n - 1)
#     denominator = np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr**2)
#     if denominator <= 0:
#         return np.nan
#     return norm.cdf(numerator / denominator)

# def evaluate_crisis_fold(df_features, tickers_all, trading_days, train_end_marker, test_start_marker,
#                           test_end_marker, fold_name):
#     train_end = nearest_trading_day(train_end_marker, trading_days, 'before')
#     test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
#     test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
#     print(f"\n=== Crisis fold {fold_name}: train đến {train_end.date()}, test {test_start.date()}->{test_end.date()} ===")

#     train_mask = (df_features['date'] >= DATA_START) & (df_features['date'] <= train_end)
#     train_df = df_features[train_mask].copy()
#     low = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[0])
#     high = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[1])
#     train_df['target'] = train_df['target_raw'].clip(low, high)

#     X, y = prepare_training_data(train_df, tickers_all,
#                                  trading_days[(trading_days>=DATA_START) & (trading_days<=train_end)],
#                                  SEQUENCE_LENGTH)
#     scaler = StandardScaler()
#     X_reshaped = X.reshape(-1, X.shape[-1])
#     scaler.fit(X_reshaped)
#     X_scaled = scaler.transform(X_reshaped).reshape(X.shape)

#     split = int(len(X_scaled) * (1 - VAL_SPLIT))
#     X_train, X_val = X_scaled[:split], X_scaled[split:]
#     y_train, y_val = y[:split], y[split:]

#     model = build_lstm((SEQUENCE_LENGTH, X.shape[2]))
#     early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
#     model.fit(X_train, y_train, validation_data=(X_val, y_val),
#               epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)

#     ret_df, _, _ = run_backtest(df_features, model, scaler, tickers_all,
#                                 test_start, test_end, trading_days,
#                                 top_frac=TOP_UNIVERSE_FRAC, top_k=TOP_K_HOLD, fee=TRADING_FEE)
#     cum = (1 + ret_df['daily_return']).cumprod()
#     rolling_max = cum.expanding().max()
#     drawdown = (cum / rolling_max) - 1
#     max_dd = drawdown.min()
#     sharpe = ret_df['daily_return'].mean() / ret_df['daily_return'].std() * np.sqrt(252) if ret_df['daily_return'].std() != 0 else 0
#     print(f"Crisis {fold_name}: Max DD = {max_dd:.4f}, Sharpe = {sharpe:.4f}")
#     if max_dd < MDD_STOP or sharpe < -1:
#         print("   >>> FAIL (MDD hoặc Sharpe không đạt)")
#     else:
#         print("   >>> PASS")
#     return max_dd, sharpe

# def main():
#     print("Đọc dữ liệu...")
#     df = load_and_prepare_data('merged_long_format.csv')
#     tickers_all = sorted(df['ticker'].unique())
#     print(f"Tổng số ticker ban đầu: {len(tickers_all)}")
#     tickers_all = [t for t in tickers_all if t not in EXCLUDED_TICKERS]
#     print(f"Số mã sau khi loại {EXCLUDED_TICKERS}: {len(tickers_all)}")
#     print(f"Các mã: {tickers_all}")

#     if datetime.now() > datetime(2026, 3, 31):
#         print("\n*** CẢNH BÁO: Crisis fold 2026 đang chạy trên dữ liệu đã biết trước. "
#               "Xác nhận rằng fold này đã được pre-register trước khi có dữ liệu Q1/2026. ***")

#     df_features = add_features(df)
#     trading_days = get_trading_days(df)

#     print("\n===== BẮT ĐẦU 5-FOLD WALK-FORWARD LIÊN TỤC =====")
#     fold_results = []

#     for i, (train_marker, test_start_marker, test_end_marker) in enumerate(FOLDS_5):
#         train_end = nearest_trading_day(train_marker, trading_days, 'before')
#         test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
#         test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
#         print(f"\nFold {i+1}: train đến {train_end.date()}, test {test_start.date()} -> {test_end.date()}")

#         train_mask = (df_features['date'] >= DATA_START) & (df_features['date'] <= train_end)
#         train_df = df_features[train_mask].copy()
#         low = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[0])
#         high = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[1])
#         train_df['target'] = train_df['target_raw'].clip(low, high)

#         X, y = prepare_training_data(train_df, tickers_all,
#                                      trading_days[(trading_days >= DATA_START) & (trading_days <= train_end)],
#                                      SEQUENCE_LENGTH)
#         scaler = StandardScaler()
#         X_reshaped = X.reshape(-1, X.shape[-1])
#         scaler.fit(X_reshaped)
#         X_scaled = scaler.transform(X_reshaped).reshape(X.shape)

#         split = int(len(X_scaled) * (1 - VAL_SPLIT))
#         X_train, X_val = X_scaled[:split], X_scaled[split:]
#         y_train, y_val = y[:split], y[split:]

#         model = build_lstm((SEQUENCE_LENGTH, X.shape[2]))
#         early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
#         model.fit(X_train, y_train, validation_data=(X_val, y_val),
#                   epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)

#         ret_df, turnover_log, corr_log = run_backtest(df_features, model, scaler, tickers_all,
#                                                        test_start, test_end, trading_days)
#         fold_results.append((ret_df, turnover_log, corr_log))

#     all_returns = concatenate_fold_returns(fold_results)
#     psr_model = compute_psr(all_returns)
#     print(f"\nPSR mô hình (5-fold liên tục): {psr_model:.4f}")

#     print("\n===== BASELINE (equal-weight universe) =====")
#     baseline_rets = []
#     for _, test_start_marker, test_end_marker in FOLDS_5:
#         test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
#         test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
#         ret_df = run_baseline_backtest(df_features, test_start, test_end, trading_days)
#         baseline_rets.append(ret_df)
#     baseline_all = pd.concat(baseline_rets, ignore_index=True)
#     psr_baseline = compute_psr(baseline_all['daily_return'])
#     print(f"PSR baseline: {psr_baseline:.4f}")

#     avg_corr_model = np.mean([c for _, _, corr_list in fold_results for _, c in corr_list]) if any(corr_list for _,_,corr_list in fold_results) else np.nan
#     wide_ret = df_features[df_features['ticker'].isin(tickers_all)].pivot_table(index='date', columns='ticker', values='return_1d').dropna()
#     if wide_ret.shape[1] >= 2:
#         avg_corr_all = wide_ret.corr().values[np.triu_indices_from(wide_ret.corr().values, k=1)].mean()
#     else:
#         avg_corr_all = np.nan
#     print(f"Tương quan trung bình {len(tickers_all)} mã: {avg_corr_all:.4f}")
#     print(f"Tương quan trung bình danh mục 3 mã: {avg_corr_model:.4f}")
#     if avg_corr_all > 0.7:
#         print("CẢNH BÁO: tương quan >0.7, N hiệu dụng thấp hơn số mã.")

#     print("\n===== CRISIS FOLDS (độc lập) =====")
#     for train_end_marker, test_start_marker, test_end_marker, name in CRISIS_FOLDS:
#         evaluate_crisis_fold(df_features, tickers_all, trading_days,
#                              train_end_marker, test_start_marker, test_end_marker, name)

#     if psr_model >= PSR_THRESHOLD and psr_model > psr_baseline:
#         print(f"\n>>> KẾT LUẬN: Mô hình có tín hiệu (PSR {psr_model:.4f} >= 0.95 và vượt baseline).")
#     else:
#         print(f"\n>>> KẾT LUẬN: Mô hình KHÔNG đạt tín hiệu (PSR model={psr_model:.4f}, baseline={psr_baseline:.4f}).")

# if __name__ == "__main__":
#     main()

Đọc dữ liệu...
Tổng số ticker ban đầu: 12
Số mã sau khi loại ['VNINDEX']: 11
Các mã: ['AGR', 'APG', 'BSI', 'CTS', 'FTS', 'HCM', 'ORS', 'SSI', 'TVS', 'VDS', 'VIX']

*** CẢNH BÁO: Crisis fold 2026 đang chạy trên dữ liệu đã biết trước. Xác nhận rằng fold này đã được pre-register trước khi có dữ liệu Q1/2026. ***

===== BẮT ĐẦU 5-FOLD WALK-FORWARD LIÊN TỤC =====

Fold 1: train đến 2018-12-28, test 2019-01-02 -> 2020-05-28

Fold 2: train đến 2020-05-28, test 2020-06-01 -> 2021-10-28

Fold 3: train đến 2021-10-28, test 2021-11-01 -> 2023-03-30

Fold 4: train đến 2023-03-30, test 2023-04-03 -> 2024-08-29

Fold 5: train đến 2024-08-29, test 2024-09-04 -> 2025-12-30

PSR mô hình (5-fold liên tục): 0.8245

===== BASELINE (equal-weight universe) =====
PSR baseline: 0.5917
Tương quan trung bình 11 mã: 0.4854
Tương quan trung bình danh mục 3 mã: 0.5112

===== CRISIS FOLDS (độc lập) =====

=== Crisis fold covid_2020: train đến 2019-12-30, test 2020-03-02->2020-06-29 ===
Crisis covid_2020: Max DD = -

Nhóm 1 — Portfolio construction / risk management (tin cậy cao nhất, vì đây là nơi thất bại thực sự xảy ra)

In [ ]:
"""
PIPELINE DỰ ĐOÁN RETURN CỔ PHIẾU VN – DỰ ÁN A (PRE-REGISTERED)
Đã sửa lỗi kế toán vốn, bổ sung tính phí khi thanh lý khẩn cấp,
sửa turnover/fee khi vol-targeting, và phân biệt lý do cash (circuit breaker vs correlation filter).
"""

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
import random
import os
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ====================== CÀI ĐẶT TÁI LẬP ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
STORE_RIDGE_CV_RESULTS = False
# ====================== THAM SỐ PRE-REGISTERED ======================
DATA_START = '2016-07-05'
TARGET_HORIZON = 5
SEQUENCE_LENGTH = 20
TOP_UNIVERSE_FRAC = 0.5
TOP_K_HOLD = 3
REBALANCE_FREQ = 5
TRADING_FEE = 0.004
WINSORIZE_QUANTILES = (0.01, 0.99)
MDD_STOP = -0.20
PSR_THRESHOLD = 0.95
EXCLUDED_TICKERS = ['VNINDEX']

# --- Tham số quản trị rủi ro (pre-registered) ---
VOL_TARGET_LOOKBACK = 60
VOL_TARGET_QUANTILE = 0.9
VOL_REDUCE_FACTOR = 0.5
CB_DRAWDOWN_THRESHOLD = -0.12
CB_RECOVERY_VOL_QUANTILE = 0.8
CORRELATION_MAX = 0.7

# LSTM
LSTM_UNITS = 64
LSTM_LAYERS = 3
DROPOUT = 0.2
LR = 0.001
BATCH_SIZE = 32
MAX_EPOCHS = 200
EARLY_STOP_PATIENCE = 10
VAL_SPLIT = 0.2

FOLDS_5 = [
    ('2018-12-31', '2019-01-02', '2020-05-29'),
    ('2020-05-29', '2020-06-01', '2021-10-29'),
    ('2021-10-29', '2021-11-01', '2023-03-31'),
    ('2023-03-31', '2023-04-03', '2024-08-30'),
    ('2024-08-30', '2024-09-02', '2025-12-31')
]
CRISIS_FOLDS = [
    ('2019-12-31', '2020-03-01', '2020-06-30', 'covid_2020'),
    ('2025-12-31', '2026-01-01', '2026-03-31', 'crash_2026')
]

# ====================== TIỆN ÍCH ======================
def load_and_prepare_data(filepath='merged_long_format.csv'):
    first_row = pd.read_csv(filepath, nrows=0)
    date_col = 'time' if 'time' in first_row.columns else 'date'
    df = pd.read_csv(filepath, parse_dates=[date_col])
    rename_map = {}
    if 'time' in df.columns:
        rename_map['time'] = 'date'
    if 'symbol' in df.columns:
        rename_map['symbol'] = 'ticker'
    if rename_map:
        df.rename(columns=rename_map, inplace=True)
    df = df[(df['is_trading'] == 1) & (df['volume'] > 0)]
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'] >= DATA_START]
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    return df

def get_trading_days(df):
    days = df['date'].drop_duplicates().sort_values().reset_index(drop=True)
    return days

def nearest_trading_day(target, days, how='before'):
    target = pd.Timestamp(target)
    if how == 'before':
        return days[days <= target].iloc[-1]
    else:
        return days[days >= target].iloc[0]

def add_features(df):
    df = df.copy()
    df['return_1d'] = df.groupby('ticker')['close'].pct_change()
    df['return_5d_lag'] = df.groupby('ticker')['close'].pct_change(5)
    df['ma5'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(5).mean())
    df['ma10'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(10).mean())
    df['ma20'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
    df['volatility_5d'] = df.groupby('ticker')['return_1d'].transform(lambda x: x.rolling(5).std())
    df['volume_ma5'] = df.groupby('ticker')['volume'].transform(lambda x: x.rolling(5).mean())
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    df['dollar_volume'] = df['close'] * df['volume']
    df['target_raw'] = df.groupby('ticker')['close'].shift(-TARGET_HORIZON) / df['close'] - 1
    return df

def prepare_training_data(df_features, tickers, train_dates, sequence_len=SEQUENCE_LENGTH):
    feature_cols = [c for c in df_features.columns if c not in
                    ['target_raw', 'target', 'dollar_volume', 'ticker', 'date']]
    X_list, y_list = [], []
    for tic in tickers:
        sub = df_features[df_features['ticker'] == tic].sort_values('date')
        sub = sub[sub['date'].isin(train_dates)]
        sub = sub.dropna(subset=feature_cols + ['target'])
        arr_f = sub[feature_cols].values
        arr_t = sub['target'].values
        for i in range(sequence_len, len(sub)):
            X_list.append(arr_f[i-sequence_len:i])
            y_list.append(arr_t[i])
    return np.array(X_list), np.array(y_list).reshape(-1, 1)

def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(LSTM_UNITS, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(DROPOUT))
    for _ in range(LSTM_LAYERS - 2):
        model.add(LSTM(LSTM_UNITS, return_sequences=True))
        model.add(Dropout(DROPOUT))
    model.add(LSTM(LSTM_UNITS, return_sequences=False))
    model.add(Dropout(DROPOUT))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=LR), loss='mse')
    return model

def filter_universe(df_features, date, trading_days, top_frac=TOP_UNIVERSE_FRAC, lookback=20):
    idx = trading_days[trading_days <= date].index[-lookback:]
    recent_dates = trading_days.iloc[idx]
    recent = df_features[df_features['date'].isin(recent_dates)]
    recent = recent[~recent['ticker'].isin(EXCLUDED_TICKERS)]
    avg_dvol = recent.groupby('ticker')['dollar_volume'].mean().sort_values(ascending=False)
    n_select = int(np.ceil(len(avg_dvol) * top_frac))
    return list(avg_dvol.head(n_select).index)

def predict_returns_for_date(model, scaler, df_features, tickers, date, sequence_len=SEQUENCE_LENGTH):
    feature_cols = [c for c in df_features.columns if c not in
                    ['target_raw', 'target', 'dollar_volume', 'ticker', 'date']]
    preds = {}
    for tic in tickers:
        sub = df_features[(df_features['ticker'] == tic) & (df_features['date'] <= date)].tail(sequence_len)
        if len(sub) < sequence_len:
            preds[tic] = -np.inf
            continue
        seq = sub[feature_cols].values
        seq_scaled = scaler.transform(seq)
        pred = model.predict(seq_scaled[np.newaxis, ...], verbose=0)[0, 0]
        preds[tic] = pred
    return preds

# ====================== QUẢN TRỊ RỦI RO (đã sửa toàn bộ lỗi) ======================
def compute_current_vol(df_features, universe, date):
    vols = []
    for tic in universe:
        row = df_features[(df_features['ticker'] == tic) & (df_features['date'] == date)]['volatility_5d']
        if not row.empty and not np.isnan(row.iloc[0]):
            vols.append(row.iloc[0])
    return np.mean(vols) if vols else 0.0

_VOL_CACHE = {}
def get_historical_vol_threshold(df_features, date, trading_days, lookback=VOL_TARGET_LOOKBACK, quantile=VOL_TARGET_QUANTILE):
    key = (date, lookback, quantile)
    if key in _VOL_CACHE:
        return _VOL_CACHE[key]
    idx = trading_days[trading_days <= date].index[-lookback:]
    dates_hist = trading_days.iloc[idx]
    hist_vols = []
    for d in dates_hist:
        uni = filter_universe(df_features, d, trading_days, top_frac=TOP_UNIVERSE_FRAC, lookback=20)
        vol = compute_current_vol(df_features, uni, d)
        if vol > 0:
            hist_vols.append(vol)
    if len(hist_vols) < 10:
        thresh = np.inf
    else:
        thresh = np.quantile(hist_vols, quantile)
    _VOL_CACHE[key] = thresh
    return thresh

def correlation_aware_selection(predictions, df_features, date, max_corr=CORRELATION_MAX, top_k=TOP_K_HOLD):
    sorted_tickers = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
    selected = []
    for tic, _ in sorted_tickers:
        if len(selected) == top_k:
            break
        if len(selected) == 0:
            selected.append(tic)
            continue
        corrs = []
        ret_tic = df_features[(df_features['ticker'] == tic) & (df_features['date'] <= date)].tail(20)['return_1d'].dropna()
        if len(ret_tic) < 5:
            continue
        for s in selected:
            ret_s = df_features[(df_features['ticker'] == s) & (df_features['date'] <= date)].tail(20)['return_1d'].dropna()
            if len(ret_s) < 5:
                continue
            common_idx = ret_tic.index.intersection(ret_s.index)
            if len(common_idx) < 5:
                continue
            corr = ret_tic[common_idx].corr(ret_s[common_idx])
            corrs.append(corr)
        if corrs and (np.mean(corrs) > max_corr):
            continue
        selected.append(tic)
    return selected

def run_backtest(df_features, model, scaler, tickers_all, start_date, end_date, trading_days,
                 top_frac=TOP_UNIVERSE_FRAC, top_k=TOP_K_HOLD, fee=TRADING_FEE,
                 rebalance_freq=REBALANCE_FREQ, initial_state=None):
    """
    initial_state: dict với 'holdings', 'cash_balance', 'prev_value', 'peak_value', 'in_cash', 'cash_reason'.
    Trả về (df_returns, turnover_log, corr_log, final_state).
    """
    sim_days = trading_days[(trading_days >= start_date) & (trading_days <= end_date)].reset_index(drop=True)
    rebalance_dates = set(sim_days[::rebalance_freq])

    if initial_state is None:
        holdings = {}
        cash_balance = 1.0
        prev_value = 1.0
        peak_value = 1.0
        in_cash = False
        cash_reason = None
    else:
        holdings = initial_state['holdings'].copy()
        cash_balance = initial_state['cash_balance']
        prev_value = initial_state['prev_value']
        peak_value = initial_state['peak_value']
        in_cash = initial_state['in_cash']
        cash_reason = initial_state.get('cash_reason', None)

    portfolio_returns = []
    turnover_log = []
    corr_log = []

    for i, date in enumerate(sim_days):
        # --- Kiểm tra phục hồi nếu đang trong cash do circuit breaker (có thể xảy ra ngay ngày đầu) ---
        if in_cash and cash_reason == 'circuit_breaker':
            universe_now = filter_universe(df_features, date, trading_days, top_frac)
            current_vol = compute_current_vol(df_features, universe_now, date)
            vol_threshold_recovery = get_historical_vol_threshold(df_features, date, trading_days,
                                                                  lookback=VOL_TARGET_LOOKBACK,
                                                                  quantile=CB_RECOVERY_VOL_QUANTILE)
            if current_vol < vol_threshold_recovery:
                in_cash = False
                cash_reason = None
                # không rebalance ngay, đợi lịch hoặc ngày mai

        # --- Tính giá trị danh mục hiện tại và daily return ---
        if i == 0:
            daily_ret = 0.0
            # Giữ nguyên prev_value, không thay đổi
        else:
            current_value = cash_balance
            for tic, shares in holdings.items():
                price_series = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                if not price_series.empty:
                    current_value += shares * price_series.iloc[0]
                else:
                    prev_price_series = df_features[(df_features['ticker']==tic) & (df_features['date']==sim_days.iloc[i-1])]['close']
                    if not prev_price_series.empty:
                        current_value += shares * prev_price_series.iloc[0]
            daily_ret = (current_value / prev_value) - 1
            prev_value = current_value

            # Cập nhật đỉnh vốn
            if current_value > peak_value:
                peak_value = current_value

            # Kiểm tra circuit breaker (chỉ khi không trong cash hoặc cash do lý do khác)
            if not in_cash:
                drawdown = (current_value / peak_value) - 1
                if drawdown <= CB_DRAWDOWN_THRESHOLD:
                    # Bán toàn bộ, chịu phí
                    fee_cost = current_value * fee  # turnover = 1.0
                    current_value -= fee_cost
                    cash_balance = current_value
                    holdings = {}
                    in_cash = True
                    cash_reason = 'circuit_breaker'
                    prev_value = current_value  # cập nhật lại prev
                    turnover_log.append((date, 1.0))
                    daily_ret = (current_value / (prev_value + fee_cost)) - 1  # tính lại daily_ret? Có thể không cần vì đã điều chỉnh prev

        # --- Rebalance nếu không bị cấm ---
        # Cho phép rebalance nếu không in_cash, hoặc cash do no_candidates (được phép thử lại)
        allow_rebalance = (not in_cash) or (cash_reason == 'no_candidates')
        do_rebalance = (date in rebalance_dates) and allow_rebalance

        if do_rebalance:
            universe = filter_universe(df_features, date, trading_days, top_frac)
            current_vol = compute_current_vol(df_features, universe, date)
            vol_threshold = get_historical_vol_threshold(df_features, date, trading_days,
                                                         lookback=VOL_TARGET_LOOKBACK,
                                                         quantile=VOL_TARGET_QUANTILE)
            invest_frac = 1.0
            if current_vol > vol_threshold:
                invest_frac = VOL_REDUCE_FACTOR

            preds = predict_returns_for_date(model, scaler, df_features, universe, date)
            selected = correlation_aware_selection(preds, df_features, date, max_corr=CORRELATION_MAX, top_k=top_k)
            effective_k = len(selected)

            # Tổng giá trị danh mục trước rebalance (đã bao gồm cash và cổ phiếu)
            total_equity = prev_value

            if effective_k == 0:
                # Không có mã nào, chuyển hết về cash
                fee_cost = total_equity * fee  # bán toàn bộ (turnover=1.0)
                cash_balance = total_equity - fee_cost
                holdings = {}
                in_cash = True
                cash_reason = 'no_candidates'
                prev_value = cash_balance
                turnover_log.append((date, 1.0))
            else:
                # Tính giá trị hiện tại của từng cổ phiếu (nếu đang nắm giữ)
                current_equity = {}
                for tic in holdings:
                    price_row = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                    if not price_row.empty:
                        current_equity[tic] = holdings[tic] * price_row.iloc[0]
                # Giá trị cash hiện tại
                cash_value = cash_balance

                # Mục tiêu phân bổ mới (tỷ trọng thật trên tổng vốn)
                target_weights = {t: invest_frac / effective_k for t in selected}
                # Tổng vốn sau phí dự kiến (tạm thời chưa biết turnover, ta cần ước lượng turnover trước)
                # Quy trình: tính target equity, suy ra giá trị mua/bán, suy ra turnover và phí, sau đó điều chỉnh.
                # Có thể làm vòng lặp nhỏ vì phí ảnh hưởng đến net_equity, nhưng với fee nhỏ (0.4%) có thể bỏ qua ảnh hưởng bậc hai.
                # Ta sẽ tính turnover dựa trên total_equity hiện tại, sau đó trừ phí vào net_equity.
                # Bước 1: target equity cho mỗi selected ticker = target_weights[t] * total_equity
                # Bước 2: diff = target_equity - current_equity.get(t, 0). Mua nếu dương, bán nếu âm.
                # Bước 3: buy_value = sum(max(diff,0)), sell_value = sum(max(-diff,0)). Cộng thêm phần bán toàn bộ các ticker không còn trong selected.
                # Bước 4: turnover = (buy_value + sell_value) / (2 * total_equity)
                # Bước 5: fee_cost = turnover * total_equity * fee
                # Bước 6: net_equity = total_equity - fee_cost
                # Bước 7: đầu tư invest_frac * net_equity vào các selected theo target_weights, số còn lại vào cash.

                target_equity = {}
                for tic in selected:
                    target_equity[tic] = target_weights[tic] * total_equity

                buy_value = 0.0
                sell_value = 0.0
                # Các ticker hiện có nhưng không còn trong selected: bán hết
                for tic in list(holdings.keys()):
                    if tic not in selected:
                        sell_value += current_equity.get(tic, 0.0)
                # Các ticker trong selected: tính diff
                for tic in selected:
                    current_val = current_equity.get(tic, 0.0)
                    target_val = target_equity[tic]
                    diff = target_val - current_val
                    if diff > 0:
                        buy_value += diff
                    elif diff < 0:
                        sell_value += -diff

                turnover = (buy_value + sell_value) / (2 * total_equity) if total_equity > 0 else 0.0
                fee_cost = turnover * total_equity * fee
                net_equity = total_equity - fee_cost

                # Phân bổ lại
                investable = invest_frac * net_equity
                cash_balance = net_equity - investable
                new_holdings = {}
                for tic in selected:
                    price_row = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                    if not price_row.empty:
                        price = price_row.iloc[0]
                        new_holdings[tic] = (investable * target_weights[tic]) / price
                holdings = new_holdings
                prev_value = net_equity  # giá trị danh mục sau rebalance
                turnover_log.append((date, turnover))

                # Nếu trước đó đang cash (do no_candidates), bây giờ đã có mã, thoát cash
                if in_cash and cash_reason == 'no_candidates':
                    in_cash = False
                    cash_reason = None

                # Tương quan danh mục
                if effective_k >= 2:
                    rets = []
                    for tic in selected:
                        sub = df_features[(df_features['ticker']==tic) & (df_features['date']<=date)].tail(20)
                        if len(sub) >= 5:
                            rets.append(sub['return_1d'].dropna().values[-20:])
                        else:
                            rets.append(np.zeros(20))
                    if rets:
                        corr_mat = np.corrcoef(rets)
                        avg_corr = (corr_mat.sum() - len(selected)) / (len(selected)*(len(selected)-1))
                        corr_log.append((date, avg_corr))

        portfolio_returns.append(daily_ret)

    res = pd.DataFrame({'date': sim_days, 'daily_return': portfolio_returns})
    final_state = {
        'holdings': holdings,
        'cash_balance': cash_balance,
        'prev_value': prev_value,
        'peak_value': peak_value,
        'in_cash': in_cash,
        'cash_reason': cash_reason
    }
    return res, turnover_log, corr_log, final_state

# Baseline (không thay đổi)
def run_baseline_backtest(df_features, start_date, end_date, trading_days, top_frac=TOP_UNIVERSE_FRAC,
                          fee=TRADING_FEE, rebalance_freq=REBALANCE_FREQ, initial_cap=1.0):
    sim_days = trading_days[(trading_days >= start_date) & (trading_days <= end_date)].reset_index(drop=True)
    rebalance_dates = set(sim_days[::rebalance_freq])
    portfolio_returns = []
    prev_value = initial_cap
    holdings = {}
    for i, date in enumerate(sim_days):
        if i == 0:
            do_rebalance = True
            daily_ret = 0.0
            portfolio_returns.append(daily_ret)
        else:
            current_value = 0.0
            for tic, shares in holdings.items():
                price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                if not price.empty:
                    current_value += shares * price.iloc[0]
            daily_ret = (current_value / prev_value) - 1
            portfolio_returns.append(daily_ret)
            prev_value = current_value
            do_rebalance = (date in rebalance_dates)
        if do_rebalance:
            universe = filter_universe(df_features, date, trading_days, top_frac)
            k = len(universe)
            if holdings:
                old_weights = {}
                total_val = prev_value
                for tic, shares in holdings.items():
                    price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                    if not price.empty:
                        old_weights[tic] = shares * price.iloc[0] / total_val
                new_weights = {t: 1.0/k for t in universe}
                all_tickers = set(list(old_weights.keys()) + list(new_weights.keys()))
                turnover = 0.5 * sum(abs(new_weights.get(t,0) - old_weights.get(t,0)) for t in all_tickers)
                fee_cost = turnover * prev_value * fee
                prev_value -= fee_cost
            holdings = {}
            for tic in universe:
                price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                if not price.empty:
                    holdings[tic] = (prev_value / k) / price.iloc[0]
    return pd.DataFrame({'date': sim_days.iloc[1:], 'daily_return': portfolio_returns[1:]})

def concatenate_fold_returns(fold_results):
    all_rets = []
    for ret_df, _, _, _ in fold_results:
        all_rets.append(ret_df['daily_return'])
    return pd.concat(all_rets, ignore_index=True)

def compute_psr(returns_series, benchmark_return=0.0):
    rets = returns_series.dropna().values
    n = len(rets)
    if n < 2:
        return np.nan
    sr = np.mean(rets) / np.std(rets, ddof=1) if np.std(rets, ddof=1) != 0 else 0.0
    skew = pd.Series(rets).skew()
    kurt = pd.Series(rets).kurtosis() + 3
    numerator = (sr - benchmark_return) * np.sqrt(n - 1)
    denominator = np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr**2)
    if denominator <= 0:
        return np.nan
    return norm.cdf(numerator / denominator)

def evaluate_crisis_fold(df_features, tickers_all, trading_days, train_end_marker, test_start_marker,
                          test_end_marker, fold_name):
    train_end = nearest_trading_day(train_end_marker, trading_days, 'before')
    test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
    test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
    print(f"\n=== Crisis fold {fold_name}: train đến {train_end.date()}, test {test_start.date()}->{test_end.date()} ===")

    train_mask = (df_features['date'] >= DATA_START) & (df_features['date'] <= train_end)
    train_df = df_features[train_mask].copy()
    low = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[0])
    high = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[1])
    train_df['target'] = train_df['target_raw'].clip(low, high)

    X, y = prepare_training_data(train_df, tickers_all,
                                 trading_days[(trading_days>=DATA_START) & (trading_days<=train_end)],
                                 SEQUENCE_LENGTH)
    scaler = StandardScaler()
    X_reshaped = X.reshape(-1, X.shape[-1])
    scaler.fit(X_reshaped)
    X_scaled = scaler.transform(X_reshaped).reshape(X.shape)

    split = int(len(X_scaled) * (1 - VAL_SPLIT))
    X_train, X_val = X_scaled[:split], X_scaled[split:]
    y_train, y_val = y[:split], y[split:]

    model = build_lstm((SEQUENCE_LENGTH, X.shape[2]))
    early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)

    # Crisis fold luôn bắt đầu với trạng thái mặc định
    ret_df, _, _, _ = run_backtest(df_features, model, scaler, tickers_all,
                                    test_start, test_end, trading_days,
                                    top_frac=TOP_UNIVERSE_FRAC, top_k=TOP_K_HOLD, fee=TRADING_FEE)
    cum = (1 + ret_df['daily_return']).cumprod()
    rolling_max = cum.expanding().max()
    drawdown = (cum / rolling_max) - 1
    max_dd = drawdown.min()
    sharpe = ret_df['daily_return'].mean() / ret_df['daily_return'].std() * np.sqrt(252) if ret_df['daily_return'].std() != 0 else 0
    print(f"Crisis {fold_name}: Max DD = {max_dd:.4f}, Sharpe = {sharpe:.4f}")
    if max_dd < MDD_STOP or sharpe < -1:
        print("   >>> FAIL (MDD hoặc Sharpe không đạt)")
    else:
        print("   >>> PASS")
    return max_dd, sharpe

def main():
    print("Đọc dữ liệu...")
    df = load_and_prepare_data('merged_long_format.csv')
    tickers_all = sorted(df['ticker'].unique())
    print(f"Tổng số ticker ban đầu: {len(tickers_all)}")
    tickers_all = [t for t in tickers_all if t not in EXCLUDED_TICKERS]
    print(f"Số mã sau khi loại {EXCLUDED_TICKERS}: {len(tickers_all)}")
    print(f"Các mã: {tickers_all}")

    if datetime.now() > datetime(2026, 3, 31):
        print("\n*** CẢNH BÁO: Crisis fold 2026 đang chạy trên dữ liệu đã biết trước. "
              "Xác nhận rằng fold này đã được pre-register trước khi có dữ liệu Q1/2026. ***")

    df_features = add_features(df)
    trading_days = get_trading_days(df)

    print("\n===== BẮT ĐẦU 5-FOLD WALK-FORWARD LIÊN TỤC (có risk management, trạng thái liên tục) =====")
    fold_results = []
    current_state = None

    for i, (train_marker, test_start_marker, test_end_marker) in enumerate(FOLDS_5):
        train_end = nearest_trading_day(train_marker, trading_days, 'before')
        test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
        test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
        print(f"\nFold {i+1}: train đến {train_end.date()}, test {test_start.date()} -> {test_end.date()}")

        train_mask = (df_features['date'] >= DATA_START) & (df_features['date'] <= train_end)
        train_df = df_features[train_mask].copy()
        low = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[0])
        high = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[1])
        train_df['target'] = train_df['target_raw'].clip(low, high)

        X, y = prepare_training_data(train_df, tickers_all,
                                     trading_days[(trading_days >= DATA_START) & (trading_days <= train_end)],
                                     SEQUENCE_LENGTH)
        scaler = StandardScaler()
        X_reshaped = X.reshape(-1, X.shape[-1])
        scaler.fit(X_reshaped)
        X_scaled = scaler.transform(X_reshaped).reshape(X.shape)

        split = int(len(X_scaled) * (1 - VAL_SPLIT))
        X_train, X_val = X_scaled[:split], X_scaled[split:]
        y_train, y_val = y[:split], y[split:]

        model = build_lstm((SEQUENCE_LENGTH, X.shape[2]))
        early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)

        ret_df, turnover_log, corr_log, final_state = run_backtest(
            df_features, model, scaler, tickers_all,
            test_start, test_end, trading_days,
            top_frac=TOP_UNIVERSE_FRAC, top_k=TOP_K_HOLD, fee=TRADING_FEE,
            initial_state=current_state
        )
        fold_results.append((ret_df, turnover_log, corr_log, final_state))
        current_state = final_state

    all_returns = concatenate_fold_returns(fold_results)
    psr_model = compute_psr(all_returns)
    print(f"\nPSR mô hình (có risk management, trạng thái liên tục): {psr_model:.4f}")

    print("\n===== BASELINE (equal-weight universe, không risk management) =====")
    baseline_rets = []
    for _, test_start_marker, test_end_marker in FOLDS_5:
        test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
        test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
        ret_df = run_baseline_backtest(df_features, test_start, test_end, trading_days)
        baseline_rets.append(ret_df)
    baseline_all = pd.concat(baseline_rets, ignore_index=True)
    psr_baseline = compute_psr(baseline_all['daily_return'])
    print(f"PSR baseline: {psr_baseline:.4f}")

    avg_corr_model = np.mean([c for _, _, corr_list, _ in fold_results for _, c in corr_list]) if any(corr_list for _,_,corr_list,_ in fold_results) else np.nan
    wide_ret = df_features[df_features['ticker'].isin(tickers_all)].pivot_table(index='date', columns='ticker', values='return_1d').dropna()
    if wide_ret.shape[1] >= 2:
        avg_corr_all = wide_ret.corr().values[np.triu_indices_from(wide_ret.corr().values, k=1)].mean()
    else:
        avg_corr_all = np.nan
    print(f"Tương quan trung bình {len(tickers_all)} mã: {avg_corr_all:.4f}")
    print(f"Tương quan trung bình danh mục 3 mã: {avg_corr_model:.4f}")
    if avg_corr_all > 0.7:
        print("CẢNH BÁO: tương quan >0.7, N hiệu dụng thấp hơn số mã.")

    print("\n===== CRISIS FOLDS (độc lập, có risk management) =====")
    for train_end_marker, test_start_marker, test_end_marker, name in CRISIS_FOLDS:
        evaluate_crisis_fold(df_features, tickers_all, trading_days,
                             train_end_marker, test_start_marker, test_end_marker, name)

    if psr_model >= PSR_THRESHOLD and psr_model > psr_baseline:
        print(f"\n>>> KẾT LUẬN: Mô hình có tín hiệu (PSR {psr_model:.4f} >= 0.95 và vượt baseline).")
    else:
        print(f"\n>>> KẾT LUẬN: Mô hình KHÔNG đạt tín hiệu (PSR model={psr_model:.4f}, baseline={psr_baseline:.4f}).")

if __name__ == "__main__":
    main()

Nhóm 2 — Giảm capacity mô hình (nhất quán với bài học vòng 2 ai-memory)

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from scipy.stats import norm
import random
import os
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ====================== CÀI ĐẶT TÁI LẬP ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# ====================== THAM SỐ PRE-REGISTERED ======================
DATA_START = '2016-07-05'
TARGET_HORIZON = 5
SEQUENCE_LENGTH = 20
TOP_UNIVERSE_FRAC = 0.5
TOP_K_HOLD = 3
REBALANCE_FREQ = 5
TRADING_FEE = 0.004
WINSORIZE_QUANTILES = (0.01, 0.99)
MDD_STOP = -0.20
PSR_THRESHOLD = 0.95
EXCLUDED_TICKERS = ['VNINDEX']

# LSTM (complex)
LSTM_UNITS = 64
LSTM_LAYERS = 3
DROPOUT = 0.2
LR = 0.001
BATCH_SIZE = 32
MAX_EPOCHS = 200
EARLY_STOP_PATIENCE = 10
VAL_SPLIT = 0.2

# LSTM nhỏ
SMALL_LSTM_UNITS = 32
SMALL_LSTM_LAYERS = 1
SMALL_DROPOUT = 0.2

FOLDS_5 = [
    ('2018-12-31', '2019-01-02', '2020-05-29'),
    ('2020-05-29', '2020-06-01', '2021-10-29'),
    ('2021-10-29', '2021-11-01', '2023-03-31'),
    ('2023-03-31', '2023-04-03', '2024-08-30'),
    ('2024-08-30', '2024-09-02', '2025-12-31')
]
CRISIS_FOLDS = [
    ('2019-12-31', '2020-03-01', '2020-06-30', 'covid_2020'),
    ('2025-12-31', '2026-01-01', '2026-03-31', 'crash_2026')
]

# ====================== TIỆN ÍCH ======================
def load_and_prepare_data(filepath='merged_long_format.csv'):
    first_row = pd.read_csv(filepath, nrows=0)
    date_col = 'time' if 'time' in first_row.columns else 'date'
    df = pd.read_csv(filepath, parse_dates=[date_col])
    rename_map = {}
    if 'time' in df.columns:
        rename_map['time'] = 'date'
    if 'symbol' in df.columns:
        rename_map['symbol'] = 'ticker'
    if rename_map:
        df.rename(columns=rename_map, inplace=True)
    df = df[(df['is_trading'] == 1) & (df['volume'] > 0)]
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'] >= DATA_START]
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    return df

def get_trading_days(df):
    days = df['date'].drop_duplicates().sort_values().reset_index(drop=True)
    return days

def nearest_trading_day(target, days, how='before'):
    target = pd.Timestamp(target)
    if how == 'before':
        return days[days <= target].iloc[-1]
    else:
        return days[days >= target].iloc[0]

def add_features(df):
    df = df.copy()
    df['return_1d'] = df.groupby('ticker')['close'].pct_change()
    df['return_5d_lag'] = df.groupby('ticker')['close'].pct_change(5)
    df['ma5'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(5).mean())
    df['ma10'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(10).mean())
    df['ma20'] = df.groupby('ticker')['close'].transform(lambda x: x.rolling(20).mean())
    df['volatility_5d'] = df.groupby('ticker')['return_1d'].transform(lambda x: x.rolling(5).std())
    df['volume_ma5'] = df.groupby('ticker')['volume'].transform(lambda x: x.rolling(5).mean())
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    df['dollar_volume'] = df['close'] * df['volume']
    df['target_raw'] = df.groupby('ticker')['close'].shift(-TARGET_HORIZON) / df['close'] - 1
    return df

def prepare_training_data(df_features, tickers, train_dates, sequence_len=SEQUENCE_LENGTH, flatten_for_linear=False):
    feature_cols = [c for c in df_features.columns if c not in
                    ['target_raw', 'target', 'dollar_volume', 'ticker', 'date']]
    X_list, y_list = [], []
    for tic in tickers:
        sub = df_features[df_features['ticker'] == tic].sort_values('date')
        sub = sub[sub['date'].isin(train_dates)]
        sub = sub.dropna(subset=feature_cols + ['target'])
        arr_f = sub[feature_cols].values
        arr_t = sub['target'].values
        for i in range(sequence_len, len(sub)):
            seq = arr_f[i-sequence_len:i]
            if flatten_for_linear:
                X_list.append(seq.flatten())
            else:
                X_list.append(seq)
            y_list.append(arr_t[i])
    X = np.array(X_list)
    y = np.array(y_list).reshape(-1, 1)
    return X, y

def build_lstm_complex(input_shape):
    model = Sequential()
    model.add(LSTM(LSTM_UNITS, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(DROPOUT))
    for _ in range(LSTM_LAYERS - 2):
        model.add(LSTM(LSTM_UNITS, return_sequences=True))
        model.add(Dropout(DROPOUT))
    model.add(LSTM(LSTM_UNITS, return_sequences=False))
    model.add(Dropout(DROPOUT))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=LR), loss='mse')
    return model

def build_lstm_small(input_shape):
    model = Sequential()
    model.add(LSTM(SMALL_LSTM_UNITS, return_sequences=False, input_shape=input_shape))
    model.add(Dropout(SMALL_DROPOUT))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=LR), loss='mse')
    return model

def train_ridge(X_train, y_train):
    # Sửa lỗi tương thích: bỏ store_cv_values
    alphas = np.logspace(-4, 4, 50)
    ridge = RidgeCV(alphas=alphas, store_cv_results=True)
    ridge.fit(X_train, y_train.ravel())
    return ridge

def filter_universe(df_features, date, trading_days, top_frac=TOP_UNIVERSE_FRAC, lookback=20):
    idx = trading_days[trading_days <= date].index[-lookback:]
    recent_dates = trading_days.iloc[idx]
    recent = df_features[df_features['date'].isin(recent_dates)]
    recent = recent[~recent['ticker'].isin(EXCLUDED_TICKERS)]
    avg_dvol = recent.groupby('ticker')['dollar_volume'].mean().sort_values(ascending=False)
    n_select = int(np.ceil(len(avg_dvol) * top_frac))
    return list(avg_dvol.head(n_select).index)

def predict_returns_for_date(model, scaler, df_features, tickers, date, model_type='lstm', sequence_len=SEQUENCE_LENGTH):
    feature_cols = [c for c in df_features.columns if c not in
                    ['target_raw', 'target', 'dollar_volume', 'ticker', 'date']]
    preds = {}
    for tic in tickers:
        sub = df_features[(df_features['ticker'] == tic) & (df_features['date'] <= date)].tail(sequence_len)
        if len(sub) < sequence_len:
            preds[tic] = -np.inf
            continue
        seq = sub[feature_cols].values
        if model_type == 'ridge':
            x_input = seq.flatten().reshape(1, -1)
            x_scaled = scaler.transform(x_input)
            pred = model.predict(x_scaled)[0]
        else:
            seq_scaled = scaler.transform(seq)
            pred = model.predict(seq_scaled[np.newaxis, ...], verbose=0)[0, 0]
        preds[tic] = pred
    return preds

def run_backtest(df_features, model, scaler, tickers_all, start_date, end_date, trading_days,
                 model_type='lstm', top_frac=TOP_UNIVERSE_FRAC, top_k=TOP_K_HOLD, fee=TRADING_FEE,
                 rebalance_freq=REBALANCE_FREQ, initial_cap=1.0):
    sim_days = trading_days[(trading_days >= start_date) & (trading_days <= end_date)].reset_index(drop=True)
    rebalance_dates = set(sim_days[::rebalance_freq])
    portfolio_returns = []
    prev_value = initial_cap
    holdings = {}
    for i, date in enumerate(sim_days):
        if i == 0:
            daily_ret = 0.0
            do_rebalance = True
        else:
            current_value = 0.0
            for tic, shares in holdings.items():
                price_series = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                if not price_series.empty:
                    current_value += shares * price_series.iloc[0]
                else:
                    prev_price_series = df_features[(df_features['ticker']==tic) & (df_features['date']==sim_days.iloc[i-1])]['close']
                    if not prev_price_series.empty:
                        current_value += shares * prev_price_series.iloc[0]
            daily_ret = (current_value / prev_value) - 1
            prev_value = current_value
            do_rebalance = (date in rebalance_dates)

        if do_rebalance:
            universe = filter_universe(df_features, date, trading_days, top_frac)
            if len(universe) < top_k:
                selected = universe
            else:
                preds = predict_returns_for_date(model, scaler, df_features, universe, date, model_type)
                sorted_preds = sorted(preds.items(), key=lambda x: x[1], reverse=True)
                selected = [t[0] for t in sorted_preds[:top_k]]

            old_weights = {}
            if holdings:
                total_val = prev_value
                for tic, shares in holdings.items():
                    price_row = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                    if not price_row.empty:
                        old_weights[tic] = shares * price_row.iloc[0] / total_val
            new_weights = {t: 1.0/top_k for t in selected}
            all_tickers = set(list(old_weights.keys()) + list(new_weights.keys()))
            turnover = 0.5 * sum(abs(new_weights.get(t,0) - old_weights.get(t,0)) for t in all_tickers)
            fee_cost = turnover * prev_value * fee
            prev_value -= fee_cost
            holdings = {}
            for tic in selected:
                price_row = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                if not price_row.empty:
                    price = price_row.iloc[0]
                    holdings[tic] = (prev_value * new_weights[tic]) / price

        portfolio_returns.append(daily_ret)

    res = pd.DataFrame({'date': sim_days, 'daily_return': portfolio_returns})
    return res

def run_baseline_backtest(df_features, start_date, end_date, trading_days, top_frac=TOP_UNIVERSE_FRAC,
                          fee=TRADING_FEE, rebalance_freq=REBALANCE_FREQ, initial_cap=1.0):
    sim_days = trading_days[(trading_days >= start_date) & (trading_days <= end_date)].reset_index(drop=True)
    rebalance_dates = set(sim_days[::rebalance_freq])
    portfolio_returns = []
    prev_value = initial_cap
    holdings = {}
    for i, date in enumerate(sim_days):
        if i == 0:
            do_rebalance = True
            daily_ret = 0.0
            portfolio_returns.append(daily_ret)
        else:
            current_value = 0.0
            for tic, shares in holdings.items():
                price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                if not price.empty:
                    current_value += shares * price.iloc[0]
            daily_ret = (current_value / prev_value) - 1
            portfolio_returns.append(daily_ret)
            prev_value = current_value
            do_rebalance = (date in rebalance_dates)
        if do_rebalance:
            universe = filter_universe(df_features, date, trading_days, top_frac)
            k = len(universe)
            if holdings:
                old_weights = {}
                total_val = prev_value
                for tic, shares in holdings.items():
                    price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                    if not price.empty:
                        old_weights[tic] = shares * price.iloc[0] / total_val
                new_weights = {t: 1.0/k for t in universe}
                all_tickers = set(list(old_weights.keys()) + list(new_weights.keys()))
                turnover = 0.5 * sum(abs(new_weights.get(t,0) - old_weights.get(t,0)) for t in all_tickers)
                fee_cost = turnover * prev_value * fee
                prev_value -= fee_cost
            holdings = {}
            for tic in universe:
                price = df_features[(df_features['ticker']==tic) & (df_features['date']==date)]['close']
                if not price.empty:
                    holdings[tic] = (prev_value / k) / price.iloc[0]
    return pd.DataFrame({'date': sim_days.iloc[1:], 'daily_return': portfolio_returns[1:]})

def concatenate_fold_returns(fold_results):
    all_rets = []
    for ret_df in fold_results:
        all_rets.append(ret_df['daily_return'])
    return pd.concat(all_rets, ignore_index=True)

def compute_psr(returns_series, benchmark_return=0.0):
    rets = returns_series.dropna().values
    n = len(rets)
    if n < 2:
        return np.nan
    sr = np.mean(rets) / np.std(rets, ddof=1) if np.std(rets, ddof=1) != 0 else 0.0
    skew = pd.Series(rets).skew()
    kurt = pd.Series(rets).kurtosis() + 3
    numerator = (sr - benchmark_return) * np.sqrt(n - 1)
    denominator = np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr**2)
    if denominator <= 0:
        return np.nan
    return norm.cdf(numerator / denominator)

def train_and_evaluate_model_type(model_type, df_features, tickers_all, trading_days):
    fold_ret_dfs = []
    for i, (train_marker, test_start_marker, test_end_marker) in enumerate(FOLDS_5):
        train_end = nearest_trading_day(train_marker, trading_days, 'before')
        test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
        test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
        print(f"  Fold {i+1}: train {train_end.date()} -> test {test_start.date()} to {test_end.date()}")

        train_mask = (df_features['date'] >= DATA_START) & (df_features['date'] <= train_end)
        train_df = df_features[train_mask].copy()
        low = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[0])
        high = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[1])
        train_df['target'] = train_df['target_raw'].clip(low, high)

        flatten = (model_type == 'ridge')
        X, y = prepare_training_data(train_df, tickers_all,
                                     trading_days[(trading_days >= DATA_START) & (trading_days <= train_end)],
                                     flatten_for_linear=flatten)
        if flatten:
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
        else:
            scaler = StandardScaler()
            orig_shape = X.shape
            X_reshaped = X.reshape(-1, X.shape[-1])
            scaler.fit(X_reshaped)
            X_scaled = scaler.transform(X_reshaped).reshape(orig_shape)

        split = int(len(X_scaled) * (1 - VAL_SPLIT))
        X_train, X_val = X_scaled[:split], X_scaled[split:]
        y_train, y_val = y[:split], y[split:]

        if model_type == 'complex':
            model = build_lstm_complex((SEQUENCE_LENGTH, X.shape[-1]))
            early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
            model.fit(X_train, y_train, validation_data=(X_val, y_val),
                      epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)
        elif model_type == 'lstm_small':
            model = build_lstm_small((SEQUENCE_LENGTH, X.shape[-1]))
            early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
            model.fit(X_train, y_train, validation_data=(X_val, y_val),
                      epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)
        elif model_type == 'ridge':
            model = train_ridge(X_train, y_train)

        ret_df = run_backtest(df_features, model, scaler, tickers_all,
                              test_start, test_end, trading_days, model_type=model_type)
        fold_ret_dfs.append(ret_df)
    all_returns = concatenate_fold_returns(fold_ret_dfs)
    psr = compute_psr(all_returns)
    return psr, fold_ret_dfs

def evaluate_crisis_for_model(model_type, df_features, tickers_all, trading_days):
    results = {}
    for train_end_marker, test_start_marker, test_end_marker, fold_name in CRISIS_FOLDS:
        train_end = nearest_trading_day(train_end_marker, trading_days, 'before')
        test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
        test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
        print(f"  Crisis {fold_name}: train {train_end.date()} -> test {test_start.date()} to {test_end.date()}")

        train_mask = (df_features['date'] >= DATA_START) & (df_features['date'] <= train_end)
        train_df = df_features[train_mask].copy()
        low = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[0])
        high = train_df['target_raw'].quantile(WINSORIZE_QUANTILES[1])
        train_df['target'] = train_df['target_raw'].clip(low, high)

        flatten = (model_type == 'ridge')
        X, y = prepare_training_data(train_df, tickers_all,
                                     trading_days[(trading_days >= DATA_START) & (trading_days <= train_end)],
                                     flatten_for_linear=flatten)
        if flatten:
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
        else:
            scaler = StandardScaler()
            orig_shape = X.shape
            X_reshaped = X.reshape(-1, X.shape[-1])
            scaler.fit(X_reshaped)
            X_scaled = scaler.transform(X_reshaped).reshape(orig_shape)

        split = int(len(X_scaled) * (1 - VAL_SPLIT))
        X_train, X_val = X_scaled[:split], X_scaled[split:]
        y_train, y_val = y[:split], y[split:]

        if model_type == 'complex':
            model = build_lstm_complex((SEQUENCE_LENGTH, X.shape[-1]))
            early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
            model.fit(X_train, y_train, validation_data=(X_val, y_val),
                      epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)
        elif model_type == 'lstm_small':
            model = build_lstm_small((SEQUENCE_LENGTH, X.shape[-1]))
            early_stop = EarlyStopping(monitor='val_loss', patience=EARLY_STOP_PATIENCE, min_delta=1e-4, restore_best_weights=True)
            model.fit(X_train, y_train, validation_data=(X_val, y_val),
                      epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=0)
        elif model_type == 'ridge':
            model = train_ridge(X_train, y_train)

        ret_df = run_backtest(df_features, model, scaler, tickers_all,
                              test_start, test_end, trading_days, model_type=model_type)
        cum = (1 + ret_df['daily_return']).cumprod()
        rolling_max = cum.expanding().max()
        drawdown = (cum / rolling_max) - 1
        max_dd = drawdown.min()
        sharpe = ret_df['daily_return'].mean() / ret_df['daily_return'].std() * np.sqrt(252) if ret_df['daily_return'].std() != 0 else 0
        results[fold_name] = {'max_dd': max_dd, 'sharpe': sharpe}
    return results

def main():
    print("Đọc dữ liệu...")
    df = load_and_prepare_data('merged_long_format.csv')
    tickers_all = sorted(df['ticker'].unique())
    tickers_all = [t for t in tickers_all if t not in EXCLUDED_TICKERS]
    print(f"Số mã: {len(tickers_all)}")
    df_features = add_features(df)
    trading_days = get_trading_days(df)

    if datetime.now() > datetime(2026, 3, 31):
        print("\n*** CẢNH BÁO: Crisis fold 2026 đang chạy trên dữ liệu đã biết trước. ***")

    # Baseline
    print("\n===== BASELINE (equal-weight universe) =====")
    baseline_rets = []
    for _, test_start_marker, test_end_marker in FOLDS_5:
        test_start = nearest_trading_day(test_start_marker, trading_days, 'after')
        test_end = nearest_trading_day(test_end_marker, trading_days, 'before')
        ret_df = run_baseline_backtest(df_features, test_start, test_end, trading_days)
        baseline_rets.append(ret_df)
    baseline_all = pd.concat(baseline_rets, ignore_index=True)
    psr_baseline = compute_psr(baseline_all['daily_return'])
    print(f"PSR baseline: {psr_baseline:.4f}\n")

    # Thí nghiệm so sánh 3 kiến trúc (chỉ để kiểm tra giả thuyết, không phải chọn mô hình)
    model_types = ['complex', 'lstm_small', 'ridge']
    results = {}
    for mtype in model_types:
        print(f"\n===== MÔ HÌNH: {mtype.upper()} =====")
        psr, fold_dfs = train_and_evaluate_model_type(mtype, df_features, tickers_all, trading_days)
        print(f"PSR {mtype}: {psr:.4f}")
        crisis = evaluate_crisis_for_model(mtype, df_features, tickers_all, trading_days)
        results[mtype] = {'psr': psr, 'crisis': crisis}
        for cname, cd in crisis.items():
            print(f"  {cname}: DD={cd['max_dd']:.4f}, Sharpe={cd['sharpe']:.4f}")

    # Bảng so sánh (không chọn winner)
    print("\n===== BẢNG SO SÁNH (thí nghiệm khoa học) =====")
    print(f"{'Model':<15} {'PSR':>8} {'Covid DD':>9} {'Covid Sharpe':>13} {'Crash2026 DD':>12} {'Crash2026 Sharpe':>16}")
    for mtype in model_types:
        c = results[mtype]['crisis']
        print(f"{mtype:<15} {results[mtype]['psr']:8.4f} {c['covid_2020']['max_dd']:9.4f} {c['covid_2020']['sharpe']:13.4f} "
              f"{c['crash_2026']['max_dd']:12.4f} {c['crash_2026']['sharpe']:16.4f}")
    print(f"{'baseline':<15} {psr_baseline:8.4f} {'---':>9} {'---':>13} {'---':>12} {'---':>16}")

    print("\nLƯU Ý: Đây là thí nghiệm kiểm tra ảnh hưởng của capacity, không phải bước chọn mô hình sản xuất.")
    print("Mọi so sánh với ngưỡng PSR 0.95 không có hiệu lực vì đã thử nhiều kiến trúc trên cùng dữ liệu.")
    print("Để chọn một kiến trúc cuối cùng, cần pre-register riêng biệt và đánh giá trên dữ liệu hoàn toàn mới.")

if __name__ == "__main__":
    main()

Đọc dữ liệu...
Số mã: 11

*** CẢNH BÁO: Crisis fold 2026 đang chạy trên dữ liệu đã biết trước. ***

===== BASELINE (equal-weight universe) =====
PSR baseline: 0.5917


===== MÔ HÌNH: COMPLEX =====
  Fold 1: train 2018-12-28 -> test 2019-01-02 to 2020-05-28
  Fold 2: train 2020-05-28 -> test 2020-06-01 to 2021-10-28
  Fold 3: train 2021-10-28 -> test 2021-11-01 to 2023-03-30
  Fold 4: train 2023-03-30 -> test 2023-04-03 to 2024-08-29


KeyboardInterrupt: 